# Titanic


## Import Library


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Load Datasets


In [ ]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")

# データの連結
test_data["Survived"] = np.nan
all_df = pd.concat([train_data, test_data], ignore_index=True, sort=False)

# データの情報
print(all_df.info())

## Observe Data


In [ ]:
# 性別ごとの生存者数
sns.barplot(x="Sex", y="Survived", data=all_df)
plt.show()

## Handling Missing Values


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# AgeをPclass, Sex, Parch,SibSpで予測する
predict_age_df = all_df[["Age", "Pclass", "Sex", "Parch", "SibSp"]]
# ワンホットエンコーディング
predict_age_df = pd.get_dummies(predict_age_df)

# 学習データとテストデータに分離して、numpyに変換
known_age = predict_age_df[predict_age_df.Age.notnull()].values
unknown_age = predict_age_df[predict_age_df.Age.isnull()].values

# 学習データを訓練データとテストデータに分離
x = known_age[:, 1:]
y = known_age[:, 0]

# ランダムフォレスト回帰モデルの作成
rf_reg = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
rf_reg.fit(x, y)

predicted_age = rf_reg.predict(unknown_age[:, 1::])
all_df.loc[all_df["Age"].isnull(), "Age"] = predicted_age

# 年齢別の生存曲線と死亡曲線
facet = sns.FacetGrid(all_df[0:890], hue="Survived", aspect=2)
facet.map(sns.kdeplot, "Age", fill=True)
facet.set(xlim=(0, all_df.loc[0:890, "Age"].max()))
facet.add_legend()
plt.show()

## Create New Feature by Name


In [ ]:
# Nameから敬称を抽出
all_df["Title"] = all_df["Name"].map(lambda x: x.split(",")[1].split(".")[0].strip())
# 軍や医療、宗教関係の敬称を「Officer」に統一
all_df["Title"].replace(["Capt", "Col", "Major", "Dr", "Rev"], "Officer", inplace=True)
# 貴族や上流階級の敬称を「Royalty」に統一
all_df["Title"].replace(["Don", "Sir", "the Countess", "Lady", "Dona"], "Royalty", inplace=True)
# 既婚女性の異なる表記を「Mrs」に統一
all_df["Title"].replace(["Mme", "Ms"], "Mrs", inplace=True)
# 未婚女性の異なる表記を「Miss」に統一
all_df["Title"].replace(["Mlle"], "Miss", inplace=True)
# オランダの若い男性の敬称を「Master」に統一
all_df["Title"].replace(["Jonkheer"], "Master", inplace=True)

sns.barplot(x="Title", y="Survived", data=all_df)

In [ ]:
# Nameから苗字を抽出
all_df["Surname"] = all_df["Name"].map(lambda x: x.split(",")[0].strip())

# 同じ苗字の人数をカウント
all_df["FamilySize"] = all_df["Surname"].map(all_df["Surname"].value_counts())

In [ ]:
# 家族で、女性か子供のグループ
female_and_child = all_df.loc[(all_df["FamilySize"] >= 2) & ((all_df["Sex"] == "female") | (all_df["Age"] <= 16))]
# 苗字(家族)ごとの生存率
female_and_child = female_and_child.groupby("Surname")["Survived"].mean()
(female_and_child.value_counts())

In [ ]:
# 家族で、男性かつ大人のグループ
male_and_adult = all_df.loc[(all_df["FamilySize"] >= 2) & ((all_df["Sex"] == "male") & (all_df["Age"] > 16))]
# 苗字(家族)ごとの生存率
male_and_adult = male_and_adult.groupby("Surname")["Survived"].mean()
(male_and_adult.value_counts())

In [ ]:
# 死亡者リスト
dead_list = set(female_and_child[female_and_child.apply(lambda x: x == 0)].index)
# 生存者リスト
survived_list = set(male_and_adult[male_and_adult.apply(lambda x: x == 1)].index)
# それぞれのリストを表示
print(dead_list)
print(survived_list)
# 死亡者リストと生存者リストをもとに、Sex, Age, Titleを典型的なものに統一
all_df.loc[
    (all_df["Survived"].isnull()) & (all_df["Surname"].apply(lambda x: x in dead_list)), ["Sex", "Age", "Title"]
] = ["male", 30.0, "Mr"]
all_df.loc[
    (all_df["Survived"].isnull()) & (all_df["Surname"].apply(lambda x: x in survived_list)), ["Sex", "Age", "Title"]
] = ["female", 5.0, "Mrs"]

## Handling Missing Fare Values


In [ ]:
# Fareの欠損値をEmbarked='S', Pclass=3 の平均値で埋める
fare = all_df.loc[(all_df["Embarked"] == "S") & (all_df["Pclass"] == 3), "Fare"].mean()
all_df["Fare"] = all_df["Fare"].fillna(fare)

## Create New Feature by SibSp, Parch


In [ ]:
all_df["Family"] = all_df["SibSp"] + all_df["Parch"] + 1
# ラベリング
all_df.loc[(all_df["Family"] > 1) & (all_df["Family"] < 5), "Family_Label"] = 2
all_df.loc[(all_df["Family"] >= 5) & (all_df["Family"] < 8) | (all_df["Family"] == 1), "Family_Label"] = 1
all_df.loc[all_df["Family"] >= 8, "Family_Label"] = 0

In [ ]:
## Create New Feature by Ticket

In [ ]:
ticket_count = dict(all_df["Ticket"].value_counts())
all_df["TicketGroup"] = all_df["Ticket"].map(ticket_count)

sns.barplot(x="TicketGroup", y="Survived", data=all_df)
plt.show()

In [ ]:
# 生存率で3つにグルーピング
all_df.loc[(all_df["TicketGroup"] >= 2) & (all_df["TicketGroup"] <= 4), "Ticket_Label"] = 2
all_df.loc[
    (all_df["TicketGroup"] >= 5) & (all_df["TicketGroup"] <= 8) | (all_df["TicketGroup"] == 1), "Ticket_Label"
] = 1
all_df.loc[(all_df["TicketGroup"] >= 11), "Ticket_Label"] = 0
sns.barplot(x="Ticket_Label", y="Survived", data=all_df)
plt.show()

## Labeling Cabin


In [ ]:
# Cabinの欠損値をUnknownで埋める
all_df["Cabin"] = all_df["Cabin"].fillna("Unknown")
# Cabinの先頭の文字をラベルにする
all_df["Cabin_Label"] = all_df["Cabin"].str.get(0)

sns.barplot(x="Cabin_Label", y="Survived", data=all_df)
plt.show()

## Handling Missing Embarked Values


In [ ]:
all_df["Embarked"] = all_df["Embarked"].fillna("S")

## Preprocessing for Model


In [ ]:
all_df = all_df[
    ["Survived", "Pclass", "Sex", "Age", "Fare", "Embarked", "Title", "Family_Label", "Cabin_Label", "Ticket_Label"]
]

# ワンホットエンコーディング
all_df = pd.get_dummies(all_df)

# 訓練データとテストデータに分離
train_df = all_df[all_df["Survived"].notnull()]
train_df["Survived"] = train_df["Survived"]
test_df = all_df[all_df["Survived"].isnull()].drop("Survived", axis=1)

# 訓練データとテストデータをnumpyに変換
x = train_df.values[:, 1:]
y = train_df.values[:, 0]
y = y.astype("int")

# 訓練データとテストデータをnumpyに変換
test_x = test_df.values

## Model Training


In [ ]:
from sklearn.feature_selection import SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_validate

select = SelectKBest(k=16)

clf = RandomForestClassifier(
    random_state=10,
    warm_start=True,
    n_estimators=26,
    max_depth=6,
    max_features="sqrt",
)
pipeline = make_pipeline(select, clf)
pipeline.fit(x, y)

# フィット結果の表示
cv_result = cross_validate(pipeline, x, y, cv=10)
print("mean_score = ", np.mean(cv_result["test_score"]))
print("mean_std = ", np.std(cv_result["test_score"]))

In [ ]:
mask = select.get_support()

list_col = list(all_df.columns[1:])

for i, j in enumerate(list_col):
    print(f"No.{i+1}: {j}={mask[i]}")

x_selected = select.transform(x)
print(f"x.shape={x.shape}, x_selected.shape={x_selected.shape}")

In [ ]:
passenger_id = test_data["PassengerId"]
predicted = pipeline.predict(test_x)

# 提出用データの作成
submission = pd.DataFrame({"PassengerId": passenger_id, "Survived": predicted.astype(np.int32)})
submission.to_csv("submission2.csv", index=False)